In [1]:
%pip install -q pandas numpy matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


# 02. Descriptive statistics

## What descriptive statistics means
It means counting and summarising before explaining. How many orders per year? Which President signed the most? Which years were busiest? How long are titles, and how long are full texts where we have them? No fancy model yet. Just the shape of the data.

## The headline shape, in one paragraph
Most orders in this dataset are old and routine. The 1930s dominate, because Franklin D. Roosevelt signed more than any other President and because Hoover and Roosevelt together issued thousands of short administrative orders. Modern Presidents sign far fewer, roughly 150 to 400 per term, not thousands. Keep that in mind whenever a chart looks like a cliff after 1945. The cliff is partly real, fewer orders, and partly a change in what an order is used for.

In [2]:
import pandas as pd, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120
from pathlib import Path
BASE = Path("..").resolve()
CH = BASE / "charts"
CH.mkdir(exist_ok=True)
DF = pd.read_csv(BASE / "data" / "processed" / "executive_orders.csv", parse_dates=["date_signed"])
print("Rows:", len(DF), "| Years:", DF["year"].min(), "to", DF["year"].max())

Rows: 10879 | Years: 1791 to 2026


In [3]:
by_year = DF.groupby("year").size()
print("Busiest years:")
print(by_year.sort_values(ascending=False).head(15).to_string())
print()
print("Quietest years with at least one order, recent:")
print(by_year[by_year.index >= 1950].sort_values().head(10).to_string())
fig, ax = plt.subplots(figsize=(12, 5))
by_year.plot(ax=ax, color="#5b7fa6")
ax.set_title("Executive orders per year in the dataset, 1791 to 2026")
ax.set_ylabel("Orders")
ax.set_xlabel("Year signed")
plt.tight_layout()
plt.savefig(CH / "02_orders_per_year.png")
plt.show()

Busiest years:
year
1933    657
1934    472
1935    384
1941    382
1940    309
1942    290
1939    287
1936    275
1930    268
1937    257
1931    249
1938    249
2025    238
1929    237
1932    215

Quietest years with at least one order, recent:
year
2024    19
2013    20
2023    24
2005    26
2006    27
2015    29
2022    29
2008    30
2002    31
2014    31


In [4]:
by_decade = DF.groupby("decade").size()
print(by_decade.to_string())
fig, ax = plt.subplots(figsize=(10, 5))
by_decade.plot(kind="bar", ax=ax, color="#5b7fa6")
ax.set_title("Orders per decade in the dataset")
ax.set_ylabel("Orders")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(CH / "02_orders_by_decade.png")
plt.show()

decade
1790       1
1820       3
1830       7
1840       8
1850      10
1860     203
1870     124
1880     188
1890     188
1900     191
1910     190
1920     304
1930    3313
1940    1801
1950     766
1960     647
1970     684
1980     511
1990     446
2000     383
2010     374
2020     537


In [5]:
by_pres = DF.groupby("president").size().sort_values(ascending=False)
print("Top Presidents in the dataset:")
print(by_pres.head(15).to_string())
fig, ax = plt.subplots(figsize=(10, 6))
top = by_pres.head(12).iloc[::-1]
ax.barh(top.index, top.values, color="#5b7fa6")
ax.set_title("Most prolific Presidents in the dataset")
ax.set_xlabel("Orders")
plt.tight_layout()
plt.savefig(CH / "02_top_presidents.png")
plt.show()
print()
print("For comparison, published reference totals: Franklin D. Roosevelt 3721, Woodrow Wilson 1803, Calvin Coolidge 1203, Theodore Roosevelt 1081.")
print("The dataset matches the Roosevelt total closely, but Wilson, Coolidge, and Theodore Roosevelt are far lower in rows, because pre-1929 rows are incomplete. See notebook 00.")

Top Presidents in the dataset:
president
Franklin D. Roosevelt    3725
Herbert Hoover           1004
Harry S. Truman           907
Donald J. Trump           512
Dwight D. Eisenhower      484
Ronald Reagan             381
William J. Clinton        364
Richard Nixon             346
Lyndon B. Johnson         325
Jimmy Carter              320
George W. Bush            291
Barack Obama              276
John F. Kennedy           214
Grover Cleveland          173
Gerald Ford               169



For comparison, published reference totals: Franklin D. Roosevelt 3721, Woodrow Wilson 1803, Calvin Coolidge 1203, Theodore Roosevelt 1081.
The dataset matches the Roosevelt total closely, but Wilson, Coolidge, and Theodore Roosevelt are far lower in rows, because pre-1929 rows are incomplete. See notebook 00.


In [6]:
print("Orders per year in office, using dataset rows and years in office from the term table:")
import sys
sys.path.insert(0, str(BASE / "scripts"))
from presidents import TERMS
from collections import defaultdict
years_in = defaultdict(float)
for name, s, e, p in TERMS:
    years_in[name] += (pd.Timestamp(e) - pd.Timestamp(s)).days / 365.25
for pres in ["Franklin D. Roosevelt", "Herbert Hoover", "Harry S. Truman", "Ronald Reagan", "Barack Obama", "Donald J. Trump", "Joseph R. Biden"]:
    n = int((DF["president"] == pres).sum())
    print(f"{pres}: {n} orders, about {years_in[pres]:.1f} years in office, about {n / years_in[pres]:.0f} per year")

Orders per year in office, using dataset rows and years in office from the term table:
Franklin D. Roosevelt: 3725 orders, about 12.1 years in office, about 308 per year
Herbert Hoover: 1004 orders, about 4.0 years in office, about 251 per year
Harry S. Truman: 907 orders, about 7.8 years in office, about 117 per year
Ronald Reagan: 381 orders, about 8.0 years in office, about 48 per year
Barack Obama: 276 orders, about 8.0 years in office, about 34 per year
Donald J. Trump: 512 orders, about 8.0 years in office, about 64 per year
Joseph R. Biden: 162 orders, about 4.0 years in office, about 40 per year


## Revocations and amendments: what the notes can and cannot say
A revocation is when a later order cancels an earlier one. An amendment is when a later order changes part of an earlier one. The Federal Register disposition notes record these links, but only for orders in the Federal Register era, and only when the note was written. So the counts below are for recent orders with notes, not for all of history. Never divide a recent revocation count by an all history total.

In [7]:
fr_rows = DF[DF["disposition_notes"].notna() & (DF["disposition_notes"] != "")]
print("Rows with disposition notes:", len(fr_rows), "out of", len(DF))
for flag in ["Revoked by a later order", "Amends an earlier order", "Amended by a later order", "Revokes an earlier order", "Superseded"]:
    n = int(fr_rows["disposition_status"].str.contains(flag, regex=False).sum())
    print(f"{flag}: {n} ({100 * n / len(fr_rows):.1f} percent of rows with notes)")
fig, ax = plt.subplots(figsize=(10, 5))
flags = ["Revoked by a later order", "Amends an earlier order", "Amended by a later order", "Revokes an earlier order", "Superseded"]
vals = [int(fr_rows["disposition_status"].str.contains(f, regex=False).sum()) for f in flags]
ax.barh(flags[::-1], vals[::-1], color="#e08a3c")
ax.set_title("Disposition flags among rows with Federal Register notes")
ax.set_xlabel("Orders")
plt.tight_layout()
plt.savefig(CH / "02_disposition_flags.png")
plt.show()

Rows with disposition notes: 1137 out of 10879
Revoked by a later order: 294 (25.9 percent of rows with notes)
Amends an earlier order: 275 (24.2 percent of rows with notes)
Amended by a later order: 164 (14.4 percent of rows with notes)
Revokes an earlier order: 205 (18.0 percent of rows with notes)
Superseded: 50 (4.4 percent of rows with notes)


## Length: titles, and full texts where we have them
Title length is a weak proxy for importance. A short title can hide a huge order, and a long title can describe a routine appointment. Use length to see change over time, not to rank importance.

In [8]:
print(DF["title_word_count"].describe().to_string())
print()
print("Median title words by decade:")
print(DF.groupby("decade")["title_word_count"].median().to_string())
fig, ax = plt.subplots(figsize=(10, 5))
DF["title_word_count"].plot(kind="hist", bins=40, ax=ax, color="#5b7fa6")
ax.set_title("Distribution of title lengths in words")
ax.set_xlabel("Words in clean title")
plt.tight_layout()
plt.savefig(CH / "02_title_length_hist.png")
plt.show()

count    10879.000000
mean        12.078408
std          8.030520
min          0.000000
25%          7.000000
50%         10.000000
75%         16.000000
max        125.000000

Median title words by decade:
decade
1790    21.0
1820     0.0
1830     0.0
1840     2.5
1850     0.0
1860     0.0
1870     5.0
1880     0.0
1890     5.0
1900     0.0
1910     8.0
1920    11.5
1930    10.0
1940    16.0
1950    17.0
1960    14.0
1970     9.0
1980     6.0
1990     8.0
2000    10.0
2010     9.0
2020     9.0


In [9]:
txt = DF[DF["text_available"] == "Full text downloaded"].copy()
txt["text_word_count"] = pd.to_numeric(txt["text_word_count"], errors="coerce")
print("Orders with full text word counts in this build:", len(txt))
if len(txt):
    print(txt["text_word_count"].describe().to_string())
    print()
    print("Longest full texts:")
    print(txt.nlargest(5, "text_word_count")[["executive_order_number", "date_signed", "president", "title_clean", "text_word_count"]].to_string(index=False))
    print()
    print("Shortest full texts:")
    print(txt.nsmallest(5, "text_word_count")[["executive_order_number", "date_signed", "president", "title_clean", "text_word_count"]].to_string(index=False))
    fig, ax = plt.subplots(figsize=(10, 5))
    txt["text_word_count"].plot(kind="hist", bins=40, ax=ax, color="#e08a3c")
    ax.set_title("Distribution of full text lengths in words, recent orders with text")
    ax.set_xlabel("Words in downloaded full text")
    plt.tight_layout()
    plt.savefig(CH / "02_fulltext_length_hist.png")
    plt.show()

Orders with full text word counts in this build: 1549
count     1549.000000
mean      1286.051001
std       1379.648311
min        133.000000
25%        571.000000
50%        974.000000
75%       1557.000000
max      19863.000000

Longest full texts:
 executive_order_number date_signed          president                                                                  title_clean  text_word_count
                14110.0  2023-10-30    Joseph R. Biden Safe, Secure, and Trustworthy Development and Use of Artificial Intelligence          19863.0
                13086.0  1998-05-27 William J. Clinton              1998 Amendments to the Manual for Courts-Martial, United States          17072.0
                13526.0  2009-12-29       Barack Obama                                     Classified National Security Information          13122.0
                14141.0  2025-01-14    Joseph R. Biden Advancing United States Leadership in Artificial Intelligence Infrastructure          11932.0
    

In [10]:
print("Longest titles in the dataset:")
print(DF.nlargest(5, "title_word_count")[["executive_order_number", "date_signed", "president", "title_clean", "title_word_count"]].to_string(index=False))
print()
print("Most prolific single years, with context:")
print("1933, Roosevelt first year and New Deal start:", int((DF["year"] == 1933).sum()))
print("1941, United States enters World War II in December:", int((DF["year"] == 1941).sum()))
print("2025, Trump second term first year:", int((DF["year"] == 2025).sum()))
print("2020, pandemic year:", int((DF["year"] == 2020).sum()))

Longest titles in the dataset:
 executive_order_number date_signed             president                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               title_clean  titl